# 농업 문서 요약 및 질의응답 — 노트북 예제

이 노트북은 `app.py`(Streamlit 앱)와 **같은 RAG 파이프라인**을 셀 단위로 실행합니다.

핵심 함수는 `app.py`에서 그대로 가져오므로, 웹 UI와 검색·답변 결과가 같습니다.

```text
PDF 로드 → 청킹 → OpenAI Embeddings → FAISS 저장
    → 질문 임베딩 → Top-K 검색 → LLM 생성
```

실행 전 `C:\env\.env`에 `OPENAI_API_KEY`가 있어야 합니다. 위에서부터 순서대로 실행하세요.


## 0. 라이브러리와 설정

`app.py`를 import하면 API 키 로드, 모델 이름, 청크 크기(800/150)가 함께 준비됩니다.


In [1]:
from app import (
    BASE_DIR,
    CHUNK_OVERLAP,
    CHUNK_SIZE,
    INDEX_DIR,
    build_faiss_index,
    compare_crops,
    embed_texts,
    gather_comparison_hits,
    get_client,
    index_signature,
    list_pdfs,
    load_and_chunk,
    load_saved_index,
    rag_answer,
    save_index,
    search,
    summarize_hits,
)

# 실습용 설정 (앱 사이드바와 같은 역할)
MAX_PAGES = 30
TOP_K = 4
REBUILD_INDEX = False  # True로 바꾸면 PDF를 다시 임베딩합니다.

print("작업 폴더:", BASE_DIR)
print("청크 크기 / 겹침:", CHUNK_SIZE, "/", CHUNK_OVERLAP)
print("인덱스 폴더:", INDEX_DIR)


작업 폴더: C:\MyCursorLab\04_스마트 농업 AI 서비스 프로토타입 개발\02_농업 문서 요약 및 질의응답 서비스
청크 크기 / 겹침: 800 / 150
인덱스 폴더: C:\MyCursorLab\04_스마트 농업 AI 서비스 프로토타입 개발\02_농업 문서 요약 및 질의응답 서비스\faiss_index


## 1. PDF 목록 확인

같은 폴더의 농업기술길잡이 PDF를 찾습니다. 과제 조건상 **2개 이상**을 인덱싱합니다.


In [2]:
pdfs = list_pdfs()
for i, p in enumerate(pdfs, start=1):
    print(f"{i:2d}. {p.name}")

SELECTED = [
    BASE_DIR / "농업기술길잡이40_딸기.PDF",
    BASE_DIR / "농업기술길잡이28_고구마.PDF",
]
SELECTED = [p for p in SELECTED if p.exists()]
if len(SELECTED) < 2:
    SELECTED = pdfs[:2]

print("\n인덱싱 대상:")
for p in SELECTED:
    print(" -", p.name)


 1. 23 상추_저화질_단면.pdf
 2. 29 토마토_저화질_단면.pdf
 3. 5 고추 최종파일_단면.pdf
 4. 농업기술길잡이105_참외.PDF
 5. 농업기술길잡이28_고구마.PDF
 6. 농업기술길잡이40_딸기.PDF
 7. 농업기술길잡이5_사과재배.PDF
 8. 수박_단면.pdf

인덱싱 대상:
 - 농업기술길잡이40_딸기.PDF
 - 농업기술길잡이28_고구마.PDF


## 2. PDF 로드 및 청킹

페이지별 텍스트를 추출한 뒤 `chunk_size=800`, `overlap=150`으로 자릅니다.

각 청크는 과제에서 요구한 Document 형태입니다.

- `page_content`: 본문
- `metadata.source`: 파일명
- `metadata.page`: 페이지 번호 (1부터)
- `metadata.chunk_id`: **같은 페이지 안에서의 조각 순서** (0, 1, 2, …)


In [3]:
documents = load_and_chunk(SELECTED, max_pages=MAX_PAGES)
print(f"청크 수: {len(documents)}")
print("파일별 청크 수:")
from collections import Counter
print(Counter(d["metadata"]["source"] for d in documents))

print("\n--- 샘플 청크 1개 ---")
sample = documents[0]
print("source   :", sample["metadata"]["source"])
print("page     :", sample["metadata"]["page"])
print("chunk_id :", sample["metadata"]["chunk_id"])
print("본문 앞 300자:\n")
print(sample["page_content"][:300])


청크 수: 82
파일별 청크 수:
Counter({'농업기술길잡이28_고구마.PDF': 44, '농업기술길잡이40_딸기.PDF': 38})

--- 샘플 청크 1개 ---
source   : 농업기술길잡이40_딸기.PDF
page     : 1
chunk_id : 0
본문 앞 300자:

발간등록번호 11-1390000-004682-01
농업기술길잡이 040
딸기
딸기는 풍미가 좋고 영양소가 풍부하여 세계적으로 호평받고 
있는 과실이다. 딸기 100그램에는 하루 섭취량에 해당하는 
비타민 C가 함유되어 있다.
040STRA WBERRY
딸기
190924_딸기_표지.indd 4 2019. 12. 2. 오후 2:46


## 3. 임베딩 + FAISS 인덱스

청크 텍스트를 `text-embedding-3-small` 벡터로 바꾼 뒤 FAISS `IndexFlatIP`에 넣습니다.

벡터는 L2 정규화하므로, 검색 점수는 **코사인 유사도**입니다.

이미 `faiss_index/`가 있고 `REBUILD_INDEX=False`이면 저장된 인덱스를 재사용합니다. (앱의 영속화와 동일)


In [4]:
saved = None if REBUILD_INDEX else load_saved_index()

if saved is not None:
    index, documents, signature = saved
    print("저장된 인덱스를 로드했습니다.")
    print("청크 수:", len(documents))
    print("설정:", signature)
else:
    print("임베딩을 시작합니다. 청크가 많으면 1~2분 걸릴 수 있습니다.")
    client = get_client()
    vectors = embed_texts(client, [d["page_content"] for d in documents])
    index = build_faiss_index(vectors)
    signature = index_signature([p.name for p in SELECTED], MAX_PAGES)
    save_index(index, documents, signature)
    print("인덱스 저장 완료")
    print("벡터 shape:", vectors.shape)
    print("FAISS ntotal:", index.ntotal)

client = get_client()
print("OpenAI 클라이언트 준비 완료")


임베딩을 시작합니다. 청크가 많으면 1~2분 걸릴 수 있습니다.
인덱스 저장 완료
벡터 shape: (82, 1536)
FAISS ntotal: 82
OpenAI 클라이언트 준비 완료


## 4. 유사도 검색 확인

질문을 같은 모델로 임베딩한 뒤, FAISS에서 Top-K 청크를 가져옵니다.

키워드가 일치하는지가 아니라 **의미가 가까운지**를 봅니다.


In [5]:
def show_hits(hits):
    for i, hit in enumerate(hits, start=1):
        meta = hit["metadata"]
        score = hit.get("score")
        score_txt = f"{score:.3f}" if isinstance(score, float) else "-"
        preview = hit["page_content"].replace("\n", " ")[:120]
        print(f"{i}) {meta['source']} / p.{meta['page']} / chunk_id={meta['chunk_id']} / 유사도 {score_txt}")
        print(f"   {preview}...")
        print()

query = "딸기 정식 시기와 주의점은?"
hits = search(client, index, documents, query, TOP_K)
print("질문:", query)
print()
show_hits(hits)


질문: 딸기 정식 시기와 주의점은?

1) 농업기술길잡이40_딸기.PDF / p.29 / chunk_id=0 / 유사도 0.359
   030 / 농업기술길잡이 딸기 / 031 01 우리나라에서 육성된 주요 품종 딸기는 화아분화와 휴면이라는 독특한 생리적 특성을 가지고 있기 때문에 품종에  따라 재배 방법에 차이가 있다. 또한 작형에 따라 정식 시기...

2) 농업기술길잡이40_딸기.PDF / p.12 / chunk_id=0 / 유사도 0.344
   개별 선과 및 포장 스티로폼 포장 (설향) 수출용 공동 선과 (매향) 난좌 포장 (금향) 딸기의 성숙 과정 토경재배 딸기 수확 고설수경재배 딸기 수확 •수확 및 포장 190923_딸기_내지.indd 11 2019. ...

3) 농업기술길잡이40_딸기.PDF / p.7 / chunk_id=0 / 유사도 0.337
   죽향 아키히메(장희) 고하 레드펄(육보) 매향 금향 설향 대왕 •딸기 주요 품종 190923_딸기_내지.indd 6 2019. 12. 2. 오후 1:15...

4) 농업기술길잡이40_딸기.PDF / p.5 / chunk_id=0 / 유사도 0.332
   농업기술길잡이  딸기 contents chapter 1 chapter 4 chapter 2 chapter 3 014 017 019 023 026 076 082 097 100 030 049 060 062 064 069...



## 5. RAG 질의응답

검색된 청크만 프롬프트에 넣고 `gpt-4o-mini`가 답합니다.

컨텍스트에 없으면 `문서에서 확인되지 않습니다`라고 답하도록 시스템 프롬프트가 고정되어 있습니다.


In [6]:
test_questions = [
    "딸기 정식 시기와 주의점은?",
    "고구마 재배 현황은?",
    "우주선은 어떻게 만드나요?",  # 문서에 없는 질문
]

for q in test_questions:
    print("=" * 60)
    print("질문:", q)
    hits = search(client, index, documents, q, TOP_K)
    answer = rag_answer(client, q, hits)
    print("\n답변:\n", answer)
    print("\n근거:")
    show_hits(hits)


질문: 딸기 정식 시기와 주의점은?

답변:
 딸기의 정식 시기는 품종에 따라 다르며, 이는 화아분화와 휴면성 정도와 관련이 매우 높습니다. 촉성재배를 할 때는 화아분화가 빠르고 휴면이 얕은 품종을 선택해야 조기 수확이 가능하다는 점에 유의해야 합니다. 반면, 화아분화가 늦은 품종을 촉성재배하는 경우에는 잎과 줄기만 무성하게 자라고 제대로 과실을 수확하지 못할 수 있습니다. 따라서 생산시기에 따라 작형별 적합한 품종을 선택하는 것이 중요합니다. (출처: 농업기술길잡이40_딸기.PDF / p.29)

근거:
1) 농업기술길잡이40_딸기.PDF / p.29 / chunk_id=0 / 유사도 0.359
   030 / 농업기술길잡이 딸기 / 031 01 우리나라에서 육성된 주요 품종 딸기는 화아분화와 휴면이라는 독특한 생리적 특성을 가지고 있기 때문에 품종에  따라 재배 방법에 차이가 있다. 또한 작형에 따라 정식 시기...

2) 농업기술길잡이40_딸기.PDF / p.12 / chunk_id=0 / 유사도 0.344
   개별 선과 및 포장 스티로폼 포장 (설향) 수출용 공동 선과 (매향) 난좌 포장 (금향) 딸기의 성숙 과정 토경재배 딸기 수확 고설수경재배 딸기 수확 •수확 및 포장 190923_딸기_내지.indd 11 2019. ...

3) 농업기술길잡이40_딸기.PDF / p.7 / chunk_id=0 / 유사도 0.337
   죽향 아키히메(장희) 고하 레드펄(육보) 매향 금향 설향 대왕 •딸기 주요 품종 190923_딸기_내지.indd 6 2019. 12. 2. 오후 1:15...

4) 농업기술길잡이40_딸기.PDF / p.5 / chunk_id=0 / 유사도 0.332
   농업기술길잡이  딸기 contents chapter 1 chapter 4 chapter 2 chapter 3 014 017 019 023 026 076 082 097 100 030 049 060 062 064 069...

질문: 고구마 재배 현황은?

답변:
 고구마 재배 현황

### 직접 질문해 보기

아래 `my_question`만 바꿔 실행하면 됩니다.


In [7]:
my_question = "고구마 병해충은?"

hits = search(client, index, documents, my_question, TOP_K)
answer = rag_answer(client, my_question, hits)

print("질문:", my_question)
print("\n답변:\n", answer)
print("\n근거:")
show_hits(hits)


질문: 고구마 병해충은?

답변:
 고구마에 발생하는 병해충으로는 검은무늬병(흑반병), 얼룩무늬병(바이러스), 덩굴쪼김병, 선충 피해, 뒷날개흰밤나방 유충이 있습니다. (출처: 농업기술길잡이28_고구마.PDF / p.7)

근거:
1) 농업기술길잡이28_고구마.PDF / p.7 / chunk_id=0 / 유사도 0.547
   7  검은무늬병(흑반병) 얼룩무늬병(바이러스) 덩굴쪼김병 덩굴쪼김병 발병 포장 선충 피해 뒷날개흰밤나방 유충 고구마에  발생하는  병해충...

2) 농업기술길잡이28_고구마.PDF / p.5 / chunk_id=0 / 유사도 0.438
   제Ⅶ장 고구마 가공 및 이용 1. 가공 이용 현황 ...............................................................................................

3) 농업기술길잡이28_고구마.PDF / p.28 / chunk_id=0 / 유사도 0.432
   28 농업기술길잡이 고구마 질은 안정성이 있으므로 가열·조리해서 이용해도 좋으나 생즙으로 이용 하는 것이 더 효과적이다. (3) 칼륨의 혈압 강하 작용 고구마에 많이 함유된 칼륨은 혈압을 내리고 스트레스를 줄이며 피...

4) 농업기술길잡이28_고구마.PDF / p.20 / chunk_id=0 / 유사도 0.423
   20 농업기술길잡이 고구마  2016년 국내 고구마 생산량은 34만 1,000톤으로 전남(광주 포함)  24.9%, 전북 18%, 경기(서울 포함) 15.6%, 충남(대전, 세종시 포함) 14% 순 으로 많았다. 최...



## 6. 문서 요약

앱의 요약 탭과 같습니다.

- **주제 요약**: 주제로 검색한 청크만 요약
- **선택 문서 요약**: 해당 PDF 앞부분 청크를 모아 요약


In [8]:
topic = "딸기 정식 시기와 주의사항"
topic_hits = search(client, index, documents, topic, TOP_K)
topic_summary = summarize_hits(client, topic, topic_hits)

print("[주제 요약]", topic)
print(topic_summary)
print("\n근거:")
show_hits(topic_hits)


[주제 요약] 딸기 정식 시기와 주의사항
딸기의 정식 시기는 품종에 따라 다르며, 이는 화아분화와 휴면성에 크게 영향을 받습니다. 촉성재배를 위해서는 화아분화가 빠르고 휴면이 얕은 품종을 선택해야 하며, 그렇지 않을 경우 잎과 줄기만 무성하게 자라 과실 수확이 어려워질 수 있습니다. 주요 품종으로는 '수홍', '매향', '설향', '고하', '대왕' 등이 있으며, 이들은 각각의 특성과 재배 조건에 맞춰 개발되었습니다. 딸기 품종 육성은 국가 기관 중심으로 이루어지며, 최근에는 생명공학 기법이 활용되고 있습니다. 

출처: 농업기술길잡이40_딸기.PDF / p.5, p.12, p.29

근거:
1) 농업기술길잡이40_딸기.PDF / p.12 / chunk_id=0 / 유사도 0.366
   개별 선과 및 포장 스티로폼 포장 (설향) 수출용 공동 선과 (매향) 난좌 포장 (금향) 딸기의 성숙 과정 토경재배 딸기 수확 고설수경재배 딸기 수확 •수확 및 포장 190923_딸기_내지.indd 11 2019. ...

2) 농업기술길잡이40_딸기.PDF / p.29 / chunk_id=0 / 유사도 0.361
   030 / 농업기술길잡이 딸기 / 031 01 우리나라에서 육성된 주요 품종 딸기는 화아분화와 휴면이라는 독특한 생리적 특성을 가지고 있기 때문에 품종에  따라 재배 방법에 차이가 있다. 또한 작형에 따라 정식 시기...

3) 농업기술길잡이40_딸기.PDF / p.5 / chunk_id=0 / 유사도 0.354
   농업기술길잡이  딸기 contents chapter 1 chapter 4 chapter 2 chapter 3 014 017 019 023 026 076 082 097 100 030 049 060 062 064 069...

4) 농업기술길잡이40_딸기.PDF / p.7 / chunk_id=0 / 유사도 0.351
   죽향 아키히메(장희) 고하 레드펄(육보) 매향 금향 설향 대왕 •딸기 주요 품종 190923_딸기_내지.indd 6 2019. 1

In [9]:
doc_name = SELECTED[0].name
sample = [
    {
        "page_content": d["page_content"],
        "metadata": d["metadata"],
        "score": None,
    }
    for d in documents
    if d["metadata"]["source"] == doc_name
][:12]

doc_summary = summarize_hits(client, f"{doc_name} 핵심 내용", sample)
print("[선택 문서 요약]", doc_name)
print(doc_summary)


[선택 문서 요약] 농업기술길잡이40_딸기.PDF
딸기는 풍미가 좋고 영양소가 풍부하여 세계적으로 인기가 있는 과일로, 100그램당 하루 섭취량에 해당하는 비타민 C를 포함하고 있다. 이 문서는 딸기의 일반 현황, 주요 품종의 특성, 생리·생태적 특성, 육묘 기술, 작형 및 재배기술, 생리장해 발생과 대책, 병해충 발생 및 방제에 대한 내용을 다루고 있다. 주요 품종으로는 설향, 매향, 금향 등이 있으며, 재배 방법으로는 시설 토경 재배, 노지 재배, 고설수경재배 등이 있다. 또한, 주요 병해충으로는 눈마름병, 윤반병, 거세미나방 등이 있으며, 생리장해로는 철 결핍, 질소 결핍 등이 있다. 

출처: 농업기술길잡이40_딸기.PDF / p.1-11


## 7. 작물 비교 요약 (선택 과제)

두 문서를 **항목마다 따로** 검색한 뒤 비교합니다.

파일명 전체를 한 질의에 넣으면 한쪽 작물 청크만 잡혀, 다른 쪽은 "문서에서 확인되지 않습니다"가 나옵니다.
아래는 `딸기 품종`, `고구마 품종`처럼 작물별로 검색합니다.


In [10]:
crop_a = SELECTED[0].name
crop_b = SELECTED[1].name
topics = ["품종", "재배 환경", "병해충"]

hits_by_topic, all_hits = gather_comparison_hits(
    client, index, documents, crop_a, crop_b, topics, TOP_K
)

print("검색된 근거 (문서별):")
for topic, hits in hits_by_topic.items():
    print(f"\n[{topic}]")
    show_hits(hits)

comparison = compare_crops(client, crop_a, crop_b, topics, hits_by_topic)
print("\n" + "=" * 60)
print(f"{crop_a}  vs  {crop_b}")
print()
print(comparison)


검색된 근거 (문서별):

[품종]
1) 농업기술길잡이40_딸기.PDF / p.7 / chunk_id=0 / 유사도 0.564
   죽향 아키히메(장희) 고하 레드펄(육보) 매향 금향 설향 대왕 •딸기 주요 품종 190923_딸기_내지.indd 6 2019. 12. 2. 오후 1:15...

2) 농업기술길잡이40_딸기.PDF / p.29 / chunk_id=0 / 유사도 0.526
   030 / 농업기술길잡이 딸기 / 031 01 우리나라에서 육성된 주요 품종 딸기는 화아분화와 휴면이라는 독특한 생리적 특성을 가지고 있기 때문에 품종에  따라 재배 방법에 차이가 있다. 또한 작형에 따라 정식 시기...

3) 농업기술길잡이40_딸기.PDF / p.1 / chunk_id=0 / 유사도 0.500
   발간등록번호 11-1390000-004682-01 농업기술길잡이 040 딸기 딸기는 풍미가 좋고 영양소가 풍부하여 세계적으로 호평받고  있는 과실이다. 딸기 100그램에는 하루 섭취량에 해당하는  비타민 C가 함유되...

4) 농업기술길잡이40_딸기.PDF / p.5 / chunk_id=0 / 유사도 0.481
   농업기술길잡이  딸기 contents chapter 1 chapter 4 chapter 2 chapter 3 014 017 019 023 026 076 082 097 100 030 049 060 062 064 069...

5) 농업기술길잡이28_고구마.PDF / p.4 / chunk_id=1 / 유사도 0.513
   ....................................... 7. 고구마 잎자루 및 끝순 나물용 재배법 ................................................... 　 09...

6) 농업기술길잡이28_고구마.PDF / p.4 / chunk_id=2 / 유사도 0.494
   ......................................... 3. 주요 품종의 특성 ........

## 8. 정리

| 단계 | 함수 | 앱에서 대응하는 위치 |
|------|------|----------------------|
| PDF 목록 | `list_pdfs` | 사이드바 문서 선택 |
| 청킹 | `load_and_chunk` | 인덱스 생성 |
| 임베딩·FAISS | `embed_texts`, `build_faiss_index` | 인덱스 생성 |
| 저장/로드 | `save_index`, `load_saved_index` | 저장된 인덱스 불러오기 |
| 검색 | `search` | 모든 탭의 근거 청크 |
| 질의응답 | `rag_answer` | 탭: 질의응답 |
| 요약 | `summarize_hits` | 탭: 문서 요약 |
| 비교 | `compare_crops` | 탭: 작물 비교 요약 |

웹 화면으로 쓰려면 터미널에서 `streamlit run app.py`를 실행하면 됩니다.
